# Kalman Filter with Time-Point Priors and Positive Coefficients

In [ ]:
import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(4)
import jax
import jax.numpy as jnp

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt
from jax import random

from bunobee.models.ssp.kalman_1d import (
    kalman_filter_1d,
    kalman_rts_smoother_1d,
 )
from bunobee.models.ssp import plot_states
from bunobee.models.ssp.utils import construct_states_prior

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# False → plain Kalman filter; True → enforce a_obs at sampled windows
USE_DISCLOSURE = True 
# number of disclosure windows to sample
N_PERIODS      = 3    
# consecutive steps per window
N_POINTS       = 10     
# disclose noise variance
DISC_OBS_SIGMA = 0.01

In [ ]:
df = pd.read_csv(
    '../resource/simple/df.csv', parse_dates=['date']
)
# for quick testing
df = df[-365:].reset_index(drop=True)

saturation_df = pd.read_csv(
    '../resource/simple/saturation_df.csv',
)
coefs_df = pd.read_csv(
    '../resource/simple/coefs_df.csv',
)

regressors = saturation_df["regressor"].tolist()
response = df["sales"].values

response_norm = response.mean()
y = np.log(response / response_norm)
y = jnp.array(y)
X = df[regressors].values
sat_arr = saturation_df.set_index("regressor").loc[regressors, "saturation"].values
X = np.log1p(X / sat_arr)
sdx = X.std(0)
sdy = y.std()
sdy_over_sdx = sdy / sdx
Z = jnp.concatenate([jnp.ones((X.shape[0], 1)), jnp.array(X)], -1)

positivity_idx = jnp.array([0] + [1] * len(regressors))
positivity_idx = positivity_idx.astype(bool)

print("y shape", y.shape)
print("Z shape", Z.shape)
print("X shape", X.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], y, label="y")

In [ ]:
n_steps = y.shape[0]
n_states = Z.shape[-1]
ssp_priors = None

if coefs_df is not None:
    coefs_lookup = coefs_df.set_index("regressor")["coef"]
    true_states = jnp.array([0.0] + [coefs_lookup[r] for r in regressors])

    if USE_DISCLOSURE:
        ssp_priors = construct_states_prior(
            n_steps=n_steps,
            n_states=n_states,
            true_states=true_states,
            regressors=regressors,
            n_periods=N_PERIODS,
            n_points=N_POINTS,
            obs_scale=DISC_OBS_SIGMA,
        )
        print("a_obs shape:", ssp_priors["a_obs"].shape)
        print("P_obs shape:", ssp_priors["P_obs"].shape)
        print("n disclosed steps:", ssp_priors.sizes["obs_point"])
    else:
        print("disclosure disabled")
else:
    print("no ground truth state available; disclosure disabled")

In [ ]:
import xarray as xr

state_labels = ["intercept"] + regressors

base_ds = xr.Dataset(
    {
        "a0":                (("state",),          np.array([0.0] + [0.1] * len(regressors))),
        "P0":                (("state",),          np.array([1.0] + [0.1] * len(regressors))),
        "sdy":               float(sdy),
        "sdy_over_sdx":      (("regressor",),      np.asarray(sdy_over_sdx)),
        "sigma_q_loc_prior": (("sigma_q_group",),  np.array([0.05, 0.01])),
        "sigma_q_scale_prior": (("sigma_q_group",), np.array([0.05, 0.01])),
    },
    coords={
        "state":         state_labels,
        "regressor":     regressors,
        "sigma_q_group": ["level", "regressor"],
    },
)

if ssp_priors is not None:
    ssp_priors = xr.merge([base_ds, ssp_priors])
else:
    null_disc = xr.Dataset(
        {
            "a_obs":          (("time", "state"), np.zeros((n_steps, n_states))),
            "P_obs":          (("time", "state"), np.full((n_steps, n_states), np.inf)),
            "obs_idx":        (("obs_point",),    np.array([], dtype=int)),
            "positivity_idx": (("state",),        np.array([False] + [True] * len(regressors))),
        },
        coords={
            "time":      np.arange(n_steps),
            "state":     state_labels,
            "obs_point": np.array([], dtype=int),
        },
    )
    ssp_priors = xr.merge([base_ds, null_disc])

ssp_priors

In [ ]:
# # Test run to verify dimensions are correct before sampling
# def test_run(ssp_priors, positivity_idx):
#     a0    = jnp.array(ssp_priors["a0"].values)
#     P0    = jnp.array(ssp_priors["P0"].values)
#     a_obs = jnp.array(ssp_priors["a_obs"].values)
#     P_obs = jnp.array(ssp_priors["P_obs"].values)

#     sigma_h = jnp.array(0.1)
#     sigma_q_raw = ssp_priors["sigma_q_loc_prior"].values  # using prior loc as a test value
#     sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], Z.shape[-1] - 1)])
#     print("sigma_q expanded:", sigma_q.shape, sigma_q)

#     lp, at, Pt, _, _, _ = kalman_filter_1d(
#         a0=a0,
#         P0=P0,
#         sigma_h=sigma_h,
#         sigma_q=sigma_q,
#         y=y,
#         Z=Z,
#         logp=True,
#         a_obs=a_obs,
#         P_obs=P_obs,
#         positivity_idx=positivity_idx,
#     )
#     print("lp:", lp)
#     print("at shape:", at.shape)
#     print("Pt shape:", Pt.shape)

# test_run(ssp_priors, positivity_idx)

In [ ]:
def make_nuts_fn(ssp_priors, y, Z):
    a0           = jnp.array(ssp_priors["a0"].values)
    P0           = jnp.array(ssp_priors["P0"].values)
    a_obs        = jnp.array(ssp_priors["a_obs"].values)
    P_obs        = jnp.array(ssp_priors["P_obs"].values)
    sdy      = float(ssp_priors["sdy"])
    sdy_over_sdx = jnp.array(ssp_priors["sdy_over_sdx"].values)
    positivity_idx = jnp.array(ssp_priors["positivity_idx"].values)

    def nuts_fn():
        # support [0, 1], mode 0.1
        sigma_h_base = numpyro.sample(
            "sigma_h_base",
            dist.Beta(2.0, 10.0),  
        )
        sigma_h = numpyro.deterministic(
            "sigma_h",
            sigma_h_base * sdy,
        )

        sigma_q_lev = numpyro.sample("sigma_q_lev_base", dist.Beta(2.0, 10.0)) * sdy * 0.1
        sigma_q_reg = numpyro.sample("sigma_q_reg_base", dist.Beta(2.0, 10.0)) * sdy_over_sdx * 0.1

        sigma_q = numpyro.deterministic(
            "sigma_q",
            jnp.concatenate([
                sigma_q_lev[None],
                sigma_q_reg,
            ]),
        )

        lp, at, Pt, vt, Ft, Kt = kalman_filter_1d(
            a0=a0,
            P0=P0,
            sigma_h=sigma_h,
            sigma_q=sigma_q,
            y=y,
            Z=Z,
            logp=True,
            a_obs=a_obs,
            P_obs=P_obs,
            positivity_idx=positivity_idx,
        )
        numpyro.factor("lp", lp)

        # Stash filter outputs so the RTS smoother can be vmapped over posterior
        # draws as a pure-JAX post-processing step after `mcmc.get_samples`.
        numpyro.deterministic("at", at)
        numpyro.deterministic("Pt", Pt)

    return nuts_fn

In [ ]:
nuts_fn = make_nuts_fn(ssp_priors, y, Z)
nuts_kernel = NUTS(nuts_fn)
mcmc = MCMC(
    nuts_kernel,
    num_warmup=400,
    num_samples=400,
    num_chains=4,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key)

In [ ]:
import arviz as az
from bunobee.models.ssp.utils import posterior_to_xarray

# Group by chain so every site has shape (n_chains, n_draws, ...).
numpyro_posteriors = mcmc.get_samples(group_by_chain=True)

# Nested vmap: outer maps over the chain axis, inner over the draw axis,
# so the smoother itself still sees a single (T, n_states) draw.
at_smooth, Pt_smooth = jax.vmap(jax.vmap(kalman_rts_smoother_1d))(
    numpyro_posteriors["at"],
    numpyro_posteriors["Pt"],
    numpyro_posteriors["sigma_q"],
)
mu = jnp.einsum("cdts,ts->cdt", at_smooth, Z)

# Smoother results flow straight into xarray; caller owns the dim/coord schema.
state_labels = ["intercept"] + regressors
posterior = posterior_to_xarray(
    {
        **numpyro_posteriors,
        "at_smooth": np.asarray(at_smooth),
        "Pt_smooth": np.asarray(Pt_smooth),
        "mu":        np.asarray(mu),
    },
    dims={
        "sigma_q":   ["state"],
        "at":        ["time", "state"],
        "Pt":        ["time", "state"],
        "at_smooth": ["time", "state"],
        "Pt_smooth": ["time", "state"],
        "mu":        ["time"],
    },
    coords={
        "state": state_labels,
        "time":  df["date"].values,
    },
    drop=["sigma_h_base", "sigma_q_lev", "sigma_q_reg"],
)

idata = az.InferenceData(posterior=posterior)
az.summary(idata, var_names=["sigma_h", "sigma_q"])

In [ ]:
state_labels = ["intercept"] + regressors
a_obs   = jnp.array(ssp_priors["a_obs"].values)   if USE_DISCLOSURE else None
P_obs   = jnp.array(ssp_priors["P_obs"].values)   if USE_DISCLOSURE else None
obs_idx = ssp_priors["obs_idx"].values.astype(int) if USE_DISCLOSURE else None

plot_states(
    posterior,
    df["date"].values,
    state_labels,
    states_key=["at", "at_smooth"],
    colors=["C0", "C1"],
    coefs_df=coefs_df,
    obs_idx=obs_idx,
    a_obs=a_obs,
    P_obs=P_obs,
    title="Filtered state coefficients — disclosure ON" if USE_DISCLOSURE else "Filtered state coefficients — disclosure OFF",
)
plt.show()

In [ ]:
# mu: (chain, draw, time); sigma_h: (chain, draw) — broadcasts across time.
mu      = posterior["mu"]
sigma_h = posterior["sigma_h"]

rng = np.random.default_rng(0)
noise = mu.copy(data=rng.standard_normal(mu.shape))
y_pred = np.exp(mu + noise * sigma_h) * response_norm

# Quantile dim comes first; values is (3, time).
yhat_lower, yhat_mid, yhat_upper = (
    y_pred.quantile([0.05, 0.5, 0.95], dim=("chain", "draw")).values
)

n = 100
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(n), response[-n:], label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(n), yhat_mid[-n:], label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(n), yhat_lower[-n:], yhat_upper[-n:], alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('State Space Model Forecast')